# Agent 1 · Perception

U-Net, masques indexés et probabilités de piste. Entraînement et essais depuis le poste de guidage.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

# Le modèle accepte la caméra grise installée, ainsi que les captures BGR.
CLASSES = ('fond', 'piste', 'vibreur', 'herbe', 'gravier', 'mur', 'vehicule_obstacle')
TAILLE = (128, 96)  # largeur, hauteur
PISTE = 1

def gris(image):
    if image is None or image.dtype != np.uint8:
        raise ValueError("Une image caméra uint8 est requise")
    if image.ndim == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    if image.ndim != 2:
        raise ValueError("Format caméra inconnu")
    return cv2.resize(image, TAILLE, interpolation=cv2.INTER_AREA)

def bloc(a, b):
    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.ReLU(),
                         nn.Conv2d(b, b, 3, padding=1), nn.ReLU())

class PerceptionUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.entree = bloc(1, 12)
        self.encodeur = bloc(12, 24)
        self.fond = bloc(24, 48)
        self.decodeur = bloc(48 + 24, 24)
        self.sortie = nn.Sequential(bloc(24 + 12, 12), nn.Conv2d(12, len(CLASSES), 1))

    def forward(self, image):
        a = self.entree(image)
        b = self.encodeur(F.max_pool2d(a, 2))
        c = self.fond(F.max_pool2d(b, 2))
        d = self.decodeur(torch.cat((F.interpolate(c, size=b.shape[-2:]), b), dim=1))
        return self.sortie(torch.cat((F.interpolate(d, size=a.shape[-2:]), a), dim=1))

    @torch.inference_mode()
    def observer(self, image):
        tenseur = torch.from_numpy(gris(image)).float()[None, None] / 255
        scores = self(tenseur).softmax(dim=1)[0]
        return scores.argmax(dim=0).numpy().astype(np.uint8), scores[PISTE].numpy()

def scene_synthetique(rng):
    w, h = TAILLE
    y, x = np.mgrid[:h, :w]
    horizon = rng.uniform(.35, .53) * h
    t = np.clip((y-horizon)/(h-horizon), 0, 1)
    courbe = rng.uniform(-.25, .25)
    centre = w * (.5 + rng.uniform(-.08,.08) + courbe * (1-t)**2)
    largeur = w * (.018 + t * rng.uniform(.34,.57))
    dist = abs(x-centre)
    sol = y > horizon
    masque = np.zeros((h,w), np.uint8)
    masque[sol] = rng.choice([3,4])
    masque[sol & (dist < largeur + 1 + 2*t)] = 2
    masque[sol & (dist < largeur)] = PISTE
    if rng.random() < .6:
        gauche = bool(rng.integers(2))
        bord = (x < centre-largeur-3) if gauche else (x > centre+largeur+3)
        masque[bord & (abs(y-horizon) < h*.055)] = 5
    if rng.random() < .65:
        yy = int(rng.uniform(.60,.82)*h)
        xx = int(centre[yy,0]+rng.uniform(-.12,.12)*w)
        largeur_objet = int(rng.integers(4,10))
        cv2.rectangle(masque, (xx-largeur_objet,yy-6), (xx+largeur_objet,yy+2), 6, -1)
    palette = np.array([165,84,188,116,143,69,30]) + rng.uniform(-17,17,7)
    image = np.clip(palette[masque] + rng.normal(0,5,(h,w)), 0,255).astype(np.uint8)
    return image, masque

def paires_reelles(dossier):
    images = sorted((dossier/'images').glob('*.png'))
    if not images:
        raise ValueError(f"Aucune image PNG dans {dossier}/images")
    paires = []
    for f in images:
        image = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        masque = cv2.imread(str(dossier/'masques'/f.name), cv2.IMREAD_UNCHANGED)
        if image is None or masque is None or masque.ndim != 2 or masque.shape != image.shape:
            raise ValueError(f"Vérifier image et masque indexé de {f.name}")
        if not np.isin(masque, [*range(len(CLASSES)),255]).all() or not (masque == PISTE).any():
            raise ValueError(f"Classes ou annotation de piste invalides pour {f}")
        paires.append((gris(image), cv2.resize(masque, TAILLE, interpolation=cv2.INTER_NEAREST)))
    return paires

def entrainer_perception(paires, epochs, seed=72, modele=None):
    torch.manual_seed(seed)
    modele = PerceptionUNet() if modele is None else modele
    optim = torch.optim.Adam(modele.parameters(), lr=.0015)
    frequences = np.zeros(len(CLASSES),np.int64)
    for _,masque in paires:
        frequences += np.bincount(masque[masque!=255],minlength=len(CLASSES))
    poids = np.sqrt(frequences.max()/np.maximum(frequences,1)).clip(1,16)
    poids = torch.from_numpy(poids).float()
    modele.train()
    for epoch in range(epochs):
        if epoch in (epochs//2,3*epochs//4):
            for groupe in optim.param_groups:
                groupe["lr"] *= .5
        for ids in torch.randperm(len(paires)).split(12):
            x, y = zip(*(paires[int(i)] for i in ids))
            logits = modele(torch.from_numpy(np.stack(x)).float()[:,None] / 255)
            loss = F.cross_entropy(logits, torch.from_numpy(np.stack(y)).long(), weight=poids, ignore_index=255)
            optim.zero_grad()
            loss.backward()
            optim.step()
    return modele.eval()

def mesurer_perception(modele, paires):
    tableau = np.zeros((len(CLASSES),len(CLASSES)), np.int64)
    for image, cible in paires:
        pred, _ = modele.observer(image)
        connu = cible != 255
        idx = cible[connu].astype(int)*len(CLASSES) + pred[connu]
        tableau += np.bincount(idx,minlength=len(CLASSES)**2).reshape(tableau.shape)
    union = tableau.sum(0) + tableau.sum(1) - tableau.diagonal()
    return {nom: float(tableau[i,i]/union[i]) if union[i] else None
            for i,nom in enumerate(CLASSES)}

In [ ]:
def annoter_session(source, destination):
    """Un polygone par surface. Les pixels non annotés gardent la valeur 255."""
    import shutil
    fichiers = sorted((source/'images').glob('*.png'))
    if not fichiers:
        raise ValueError(f"Aucune capture PNG dans {source}/images")
    for nom in ('images','masques'):
        (destination/nom).mkdir(exist_ok=True,parents=True)
    fenetre = 'OUARTI annotations'
    cv2.namedWindow(fenetre)
    points = []
    def clic(event,x,y,flags,param):
        if event==cv2.EVENT_LBUTTONDOWN:
            points.append((x,y))
    cv2.setMouseCallback(fenetre,clic)
    try:
        for f in fichiers:
            cible = destination/'masques'/f.name
            if cible.exists():
                continue
            image = cv2.imread(str(f))
            if image is None:
                raise ValueError(f"Image illisible {f}")
            masque = np.full(image.shape[:2],255,np.uint8)
            classe = PISTE
            points.clear()
            while True:
                vue = image.copy()
                connu = masque!=255
                teintes = np.array([[90,90,90],[50,210,60],[20,100,255],[50,150,100],
                                    [40,160,205],[210,160,50],[40,40,230]],np.uint8)
                vue[connu] = (.45*vue[connu]+.55*teintes[masque[connu]]).astype(np.uint8)
                if points:
                    cv2.polylines(vue,[np.array(points)],False,(255,0,255),1)
                cv2.putText(vue,CLASSES[classe],(4,16),0,.5,(255,255,255),1)
                cv2.imshow(fenetre,vue)
                touche = cv2.waitKey(40)&255
                if touche==27 or cv2.getWindowProperty(fenetre,cv2.WND_PROP_VISIBLE)<1:
                    return
                if ord('0')<=touche<=ord('6'):
                    classe = touche-ord('0')
                elif touche in (10,13) and len(points)>=3:
                    cv2.fillPoly(masque,[np.asarray(points)],classe)
                    points.clear()
                elif touche==ord('u') and points:
                    points.pop()
                elif touche==ord('n'):
                    break
                elif touche==ord('s') and not points and (masque==PISTE).any() and ((masque!=255)&(masque!=PISTE)).any():
                    shutil.copyfile(f,destination/'images'/f.name)
                    if not cv2.imwrite(str(cible),masque):
                        raise OSError(f"Écriture impossible {cible}")
                    break
    finally:
        cv2.destroyAllWindows()

In [ ]:
def tester_a1(modele):
    rng = np.random.default_rng(9876)
    paires = [scene_synthetique(rng) for _ in range(48)]
    iou = mesurer_perception(modele,paires)
    masque,p = modele.observer(paires[0][0])
    assert masque.shape==p.shape==(96,128) and np.isfinite(p).all()
    assert iou['piste']>=.75, f"IoU piste synthétique insuffisante : {iou}"
    assert iou['vehicule_obstacle']>=.35, f"Détection synthétique des obstacles insuffisante : {iou}"
    return {'images':len(paires),'iou_synthetique':iou}